# 📈 Stock-PCMP — Probabilistic Capital Markets Pipeline
## Hybrid BiLSTM + ARIMA + XGBoost Ensemble with Market Physics Constraints

---

| Module | Description |
|--------|-------------|
| **M1** | Market Physics Engine — Volatility Bounds, Momentum & Trend Constraints |
| **M2** | OHLCV Data Loader — Sliding-Window with Technical Indicators |
| **M3** | BiLSTM PINN Core — Physics-Informed Loss + MC-Dropout |
| **M4** | Probabilistic Output — VaR, CVaR, Sharpe, Max Drawdown |
| **M5** | Hybrid Ensemble — BiLSTM + ARIMA + XGBoost Stacking |

> **Architecture:** BiLSTM encoder → MC-Dropout head → Volatility-constrained output  
> fused with ARIMA (trend/seasonality) and XGBoost (nonlinear feature interactions)

**Adapts the PCMP solar pipeline architecture directly to financial time series.**

## 📦 Imports & Setup

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

# Data & ML
import yfinance as yf
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# ARIMA
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.stattools import adfuller

# XGBoost
import xgboost as xgb

# Plotting
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
plt.style.use('dark_background')

from typing import Tuple, Dict, List, Optional
import json, math

torch.manual_seed(42)
np.random.seed(42)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')
print('All imports OK ✓')

---
## 🔬 Module 1 — Market Physics Engine

Analogous to the solar **PhysicsConstraintEngine**, this module enforces financial boundary conditions that the neural network cannot violate:

| Constraint | Formula | Meaning |
|------------|---------|--------|
| **Volatility Bound** | `ΔP ≤ μ + k·σ` | Price move bounded by historical vol |
| **Momentum Band** | RSI ∈ [0, 100] | Prevents physically impossible RSI |
| **ATR Circuit Breaker** | `|ΔP| ≤ 3·ATR` | No prediction beyond 3× average range |
| **Trend Inertia** | EMA crossover penalty | Penalises reversals against strong trend |

In [ ]:
class MarketPhysicsEngine:
    """
    Computes financial boundary conditions for stock price prediction.
    Analogous to PhysicsConstraintEngine in the solar PCMP pipeline.

    Enforces:
      - Volatility ceiling  (σ-based max daily move)
      - ATR circuit breaker (no prediction > k·ATR from last close)
      - RSI momentum band   (RSI in [0,100], overextension penalty)
      - Trend inertia score (EMA alignment signal)
    """

    K_SIGMA:   float = 3.0    # max allowable move in σ units
    K_ATR:     float = 3.0    # circuit breaker: k × ATR
    RSI_OB:    float = 70.0   # RSI overbought threshold
    RSI_OS:    float = 30.0   # RSI oversold threshold

    # ── Volatility Bound ────────────────────────────────────────
    def volatility_bound(self, close: np.ndarray, window: int = 20) -> float:
        """
        Returns the maximum allowable absolute price change for the next step.
        Based on rolling σ of log-returns × K_SIGMA.

        Args:
            close  : 1-D array of closing prices
            window : look-back for σ estimation
        Returns:
            max_move : absolute price ceiling (same units as `close`)
        """
        if len(close) < 2:
            return float('inf')
        log_ret = np.diff(np.log(close + 1e-9))
        sigma   = log_ret[-window:].std() if len(log_ret) >= window else log_ret.std()
        # convert log-return σ back to price σ
        max_move = close[-1] * (np.exp(self.K_SIGMA * sigma) - 1)
        return float(max_move)

    # ── ATR (Average True Range) ─────────────────────────────────
    def atr(self, high: np.ndarray, low: np.ndarray,
            close: np.ndarray, period: int = 14) -> float:
        """
        Wilder ATR — the canonical volatility ruler in technical analysis.
        Used as a circuit breaker: model outputs capped at K_ATR × ATR.
        """
        n = min(period, len(close) - 1)
        tr = np.maximum(
            high[-n:] - low[-n:],
            np.maximum(
                np.abs(high[-n:] - close[-n-1:-1]),
                np.abs(low[-n:]  - close[-n-1:-1])
            )
        )
        return float(tr.mean())

    # ── RSI ──────────────────────────────────────────────────────
    def rsi(self, close: np.ndarray, period: int = 14) -> float:
        """
        Wilder RSI ∈ [0, 100].  Returns 50.0 if insufficient data.
        Analogous to the clear_sky_index —  a physics-derived ceiling.
        """
        if len(close) < period + 1:
            return 50.0
        delta  = np.diff(close[-period-1:])
        gain   = delta[delta > 0].mean() if (delta > 0).any() else 1e-9
        loss   = -delta[delta < 0].mean() if (delta < 0).any() else 1e-9
        rs     = gain / loss
        return float(100 - 100 / (1 + rs))

    # ── EMA Trend Inertia ────────────────────────────────────────
    def trend_inertia(self, close: np.ndarray,
                       fast: int = 12, slow: int = 26) -> float:
        """
        Returns a signed trend strength ∈ [-1, +1].
        +1 → strong uptrend (fast EMA >> slow EMA)
        -1 → strong downtrend
        Analogous to the thermal_degradation coefficient in solar.
        """
        def ema(arr, span):
            k = 2 / (span + 1)
            e = arr[0]
            for v in arr[1:]:
                e = v * k + e * (1 - k)
            return e

        if len(close) < slow:
            return 0.0
        ema_f = ema(close[-slow:], fast)
        ema_s = ema(close[-slow:], slow)
        spread = (ema_f - ema_s) / (ema_s + 1e-9)
        return float(np.tanh(spread * 10))  # squash to [-1,1]

    # ── Penalty (used in physics loss) ──────────────────────────
    def physics_penalty(
        self,
        pred_price:  torch.Tensor,   # (B, 1)
        last_close:  torch.Tensor,   # (B, 1)
        vol_ceiling: torch.Tensor,   # (B, 1)  from volatility_bound()
    ) -> torch.Tensor:
        """
        Penalises predicted prices that exceed the volatility ceiling.
        This is the stock equivalent of the solar physics loss term:
          L_phys = λ · E[ReLU(ŷ - clear_sky_limit)]
        Here:
          L_phys = λ · E[ReLU(|Δprice| - vol_ceiling)]
        """
        delta = (pred_price - last_close).abs()
        return F.relu(delta - vol_ceiling).mean()


# ── Quick sanity check ───────────────────────────────────────────
eng = MarketPhysicsEngine()

# Synthetic OHLCV for a ~₹1500 stock
np.random.seed(0)
prices = 1500 + np.cumsum(np.random.randn(100) * 10)
highs  = prices + np.abs(np.random.randn(100) * 5)
lows   = prices - np.abs(np.random.randn(100) * 5)

print('=== Module 1 — Market Physics Engine ===')
print(f'  Volatility ceiling (20d)  : ₹{eng.volatility_bound(prices):.2f}')
print(f'  ATR (14d)                 : ₹{eng.atr(highs, lows, prices):.2f}')
print(f'  RSI (14d)                 : {eng.rsi(prices):.1f}')
print(f'  Trend inertia             : {eng.trend_inertia(prices):+.4f}')
print('Module 1 ✓')

---
## 📂 Module 2 — OHLCV Data Loader

Analogous to **ClimateWindowDataset**, this module:
- Fetches OHLCV data via `yfinance`
- Engineers 15 technical indicator features (RSI, MACD, Bollinger Bands, etc.)
- Builds sliding-window tensors for the BiLSTM
- Also builds tabular feature matrices for XGBoost

**Features per timestep (12 channels):**
`[Close_norm, Open, High, Low, Volume, RSI, MACD, BB_upper, BB_lower, EMA12, EMA26, ATR]`

In [ ]:
class StockWindowDataset(Dataset):
    """
    Sliding-window OHLCV dataset with technical indicators.
    Analogous to ClimateWindowDataset in the PCMP solar pipeline.

    Features per timestep (12 channels):
        [close_norm, open_norm, high_norm, low_norm, volume_norm,
         rsi, macd_norm, bb_upper_norm, bb_lower_norm,
         ema12_norm, ema26_norm, atr_norm]

    Each sample yields:
        (feature_window, next_close_target, vol_ceiling_tensor)
    """

    FEATURE_NAMES = [
        'close','open','high','low','volume',
        'rsi','macd','bb_upper','bb_lower',
        'ema12','ema26','atr'
    ]
    N_FEATURES = len(FEATURE_NAMES)

    def __init__(
        self,
        x:         torch.Tensor,   # (N, N_FEATURES) — normalised features
        y:         torch.Tensor,   # (N,) — normalised close targets
        vol_ceil:  torch.Tensor,   # (N,) — per-timestep volatility ceiling
        window:    int = 60,
    ):
        self.x        = x
        self.y        = y
        self.vol_ceil = vol_ceil
        self.window   = window

    def __len__(self) -> int:
        return len(self.x) - self.window

    def __getitem__(self, idx: int):
        feat_win = self.x[idx : idx + self.window]          # (W, F)
        target   = self.y[idx + self.window]                # scalar
        ceil_val = self.vol_ceil[idx + self.window]         # scalar
        return feat_win, target.unsqueeze(0), ceil_val.unsqueeze(0)

    # ── Static feature engineering ─────────────────────────────
    @staticmethod
    def from_dataframe(
        df:       pd.DataFrame,   # must have Open,High,Low,Close,Volume columns
        window:   int = 60,
        engine:   MarketPhysicsEngine = None,
    ) -> 'StockWindowDataset':
        """
        Build a StockWindowDataset directly from a yfinance DataFrame.
        All features are MinMax-normalised to [0, 1].
        """
        eng = engine or MarketPhysicsEngine()
        df  = df.copy().dropna().reset_index(drop=True)

        close  = df['Close'].values.astype(float)
        opens  = df['Open'].values.astype(float)
        high   = df['High'].values.astype(float)
        low    = df['Low'].values.astype(float)
        vol    = df['Volume'].values.astype(float)
        N      = len(close)

        # ── Technical indicators ────────────────────────────────
        def ema_series(arr, span):
            k, out, e = 2/(span+1), [], arr[0]
            for v in arr:
                e = v*k + e*(1-k); out.append(e)
            return np.array(out)

        ema12  = ema_series(close, 12)
        ema26  = ema_series(close, 26)
        macd   = ema12 - ema26

        rsi_arr = np.array([eng.rsi(close[:i+1]) for i in range(N)]) / 100.0

        # Bollinger Bands (20-day)
        bb_u, bb_l = np.zeros(N), np.zeros(N)
        for i in range(N):
            sl  = close[max(0, i-19):i+1]
            m   = sl.mean(); s = sl.std() + 1e-9
            bb_u[i] = m + 2*s
            bb_l[i] = m - 2*s

        # ATR series
        atr_arr = np.array([
            eng.atr(high[:i+1], low[:i+1], close[:i+1])
            for i in range(N)
        ])

        # Volatility ceiling per step
        vol_ceil_raw = np.array([
            eng.volatility_bound(close[:i+1])
            for i in range(N)
        ])

        # ── Assemble raw feature matrix ─────────────────────────
        raw = np.stack([
            close, opens, high, low, vol,
            rsi_arr * 100,   # un-norm RSI for the scaler
            macd, bb_u, bb_l,
            ema12, ema26, atr_arr
        ], axis=1)  # (N, 12)

        raw = np.nan_to_num(raw, nan=0.0, posinf=0.0, neginf=0.0)

        # ── MinMax normalise ────────────────────────────────────
        scaler = MinMaxScaler()
        raw_norm = scaler.fit_transform(raw)

        # Target = normalised close (column 0)
        close_scaler = MinMaxScaler()
        close_norm = close_scaler.fit_transform(close.reshape(-1,1)).flatten()

        # Normalise vol_ceil to [0,1] so it pairs with normalised predictions
        vc_scaler = MinMaxScaler()
        vc_norm   = vc_scaler.fit_transform(
            vol_ceil_raw.reshape(-1,1)).flatten()

        x_t    = torch.tensor(raw_norm,   dtype=torch.float32)
        y_t    = torch.tensor(close_norm, dtype=torch.float32)
        vc_t   = torch.tensor(vc_norm,    dtype=torch.float32)

        dataset = StockWindowDataset(x_t, y_t, vc_t, window=window)
        # Store scalers for inverse-transform at inference
        dataset.close_scaler = close_scaler
        dataset.scaler       = scaler
        dataset.raw_features = raw          # for XGBoost (un-normalised)
        dataset.close_raw    = close
        return dataset


# ── Sanity check with synthetic data ──────────────────────────
print('=== Module 2 — OHLCV Data Loader ===')
N_FAKE = 400
fake_close = 1500 + np.cumsum(np.random.randn(N_FAKE)*8)
fake_df = pd.DataFrame({
    'Open':   fake_close * (1 + np.random.randn(N_FAKE)*0.002),
    'High':   fake_close * (1 + np.abs(np.random.randn(N_FAKE)*0.005)),
    'Low':    fake_close * (1 - np.abs(np.random.randn(N_FAKE)*0.005)),
    'Close':  fake_close,
    'Volume': np.abs(np.random.randn(N_FAKE) * 1e6 + 5e6),
})

ds  = StockWindowDataset.from_dataframe(fake_df, window=60)
dl  = DataLoader(ds, batch_size=16, shuffle=True)
fb, tb, vcb = next(iter(dl))

print(f'  Dataset length        : {len(ds)} windows')
print(f'  Feature window shape  : {tuple(fb.shape)}  → (batch, time_steps, features)')
print(f'  Target shape          : {tuple(tb.shape)}')
print(f'  Vol ceiling shape     : {tuple(vcb.shape)}')
print(f'  Feature names         : {StockWindowDataset.FEATURE_NAMES}')
print('Module 2 ✓')

---
## 🧠 Module 3 — BiLSTM PINN Core

**Architecture (identical to PCMP solar M3, re-targeted to price):**
```
Input (B, W, 12)
    └─► BiLSTM (2 layers, H=128, bidirectional)
            └─► Last hidden concat (B, 256)
                    └─► MC-Dropout → Linear(256→128) → ReLU
                            └─► MC-Dropout → Linear(128→64) → ReLU
                                    └─► Linear(64→1) → Sigmoid
                                            └─► Predicted price ∈ [0,1]
```

**Physics-Informed Loss (same structure as solar PCMP):**
$$\mathcal{L} = \underbrace{\text{MSE}(\hat{y}, y)}_{\text{data fidelity}} + \underbrace{\lambda \cdot \mathbb{E}[\text{ReLU}(|\Delta P| - \sigma_{\text{ceil}})]|}_{\text{volatility constraint}}$$

In [ ]:
class StockPINNCore(nn.Module):
    """
    BiLSTM + Dense PINN for stock price prediction.
    Direct adaptation of PINNCore from the PCMP solar pipeline.

    Architectural contract
    ──────────────────────
    Input  : (B, W, F)  — batch × window × features (12 channels)
    Output : (B, 1)     — predicted normalised closing price ∈ [0,1]
    """

    def __init__(
        self,
        n_features: int   = 12,   # input feature channels
        hidden:     int   = 128,  # LSTM hidden dim
        n_layers:   int   = 2,    # stacked LSTM layers
        dropout:    float = 0.3,  # MC-Dropout probability
    ) -> None:
        super().__init__()

        # ── BiLSTM encoder (same as solar PINNCore) ────────────
        self.rnn = nn.LSTM(
            input_size    = n_features,
            hidden_size   = hidden,
            num_layers    = n_layers,
            batch_first   = True,
            bidirectional = True,
            dropout       = dropout if n_layers > 1 else 0.0,
        )

        # ── Dense head with MC-Dropout ─────────────────────────
        D = hidden * 2  # bidirectional doubles the hidden dim
        self.fc = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(D, D // 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(D // 2, D // 4),
            nn.ReLU(),
            nn.Linear(D // 4, 1),
            nn.Sigmoid(),    # price is normalised [0,1]
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        x : (B, W, F)
        Returns: (B, 1)
        """
        # BiLSTM → take last time step of both directions
        out, _ = self.rnn(x)          # (B, W, 2H)
        last   = out[:, -1, :]        # (B, 2H)
        return self.fc(last)          # (B, 1)

    # ── Physics-Informed Loss ───────────────────────────────────
    def physics_loss(
        self,
        pred:      torch.Tensor,   # (B,1)  predicted normalised price
        target:    torch.Tensor,   # (B,1)  actual normalised price
        vol_ceil:  torch.Tensor,   # (B,1)  normalised volatility ceiling
        last_close: torch.Tensor,  # (B,1)  last observed normalised close
        lam:       float = 5.0,    # physics penalty weight
    ) -> torch.Tensor:
        """
        L = MSE(pred, target)  +  λ · E[ ReLU(|Δprice| − vol_ceiling) ]

        Directly mirrors the solar physics loss:
          L = MSE + λ · E[ ReLU(ŷ − clear_sky_limit) ]
        """
        data_loss = F.mse_loss(pred, target)
        delta     = (pred - last_close).abs()
        phys_pen  = F.relu(delta - vol_ceil).mean()
        return data_loss + lam * phys_pen


# ── Sanity check ─────────────────────────────────────────────
W, F, B = 60, StockWindowDataset.N_FEATURES, 16

model = StockPINNCore(n_features=F, hidden=128, n_layers=2, dropout=0.3).to(DEVICE)
opt   = torch.optim.Adam(model.parameters(), lr=1e-3)

model.train()
feat_b, tgt_b, vc_b = fb.to(DEVICE), tb.to(DEVICE), vcb.to(DEVICE)
# last close = feature[:, -1, 0] (close channel = feature 0)
last_c = feat_b[:, -1, 0:1]

pred_b = model(feat_b)
loss   = model.physics_loss(pred_b, tgt_b, vc_b, last_c, lam=5.0)
loss.backward()
opt.step()

print('=== Module 3 — BiLSTM PINN Core ===')
print(f'  Total parameters      : {sum(p.numel() for p in model.parameters()):,}')
print(f'  Forward output shape  : {tuple(pred_b.shape)}  → (batch, 1)')
print(f'  Physics-informed loss : {loss.item():.6f}')
print('Module 3 ✓')

---
## 📊 Module 4 — Probabilistic & Risk Output Layer

Analogous to **ProbabilisticEvaluator** in the solar pipeline, but outputs financial risk metrics instead of LCOE:

| Metric | Formula | Meaning |
|--------|---------|--------|
| **P50** | 50th pct of MC distribution | Expected price |
| **P90** | 90th pct | Bullish scenario |
| **P10** | 10th pct | Bearish scenario |
| **VaR (95%)** | 5th pct of returns distribution | Daily Value-at-Risk |
| **CVaR (95%)** | E[loss \| loss > VaR] | Expected Shortfall |
| **Sharpe (est.)** | (P50−last) / σ_mc | Signal-to-noise |
| **Max Drawdown** | max(peak−trough)/peak | Worst peak-to-trough |

In [ ]:
class ProbabilisticEvaluator:
    """
    Monte Carlo Dropout inference + financial risk metrics.
    Analogous to ProbabilisticEvaluator in the solar PCMP pipeline.

    The key difference: we output price distributions and risk metrics
    (VaR, CVaR, Sharpe) instead of energy metrics (P50/P90/LCOE).
    """

    def __init__(self, model: StockPINNCore, n_samples: int = 200):
        self.model     = model
        self.n_samples = n_samples

    # ── MC-Dropout inference ──────────────────────────────────
    def monte_carlo_samples(
        self, x: torch.Tensor  # (1, W, F) or (B, W, F)
    ) -> torch.Tensor:         # (n_samples, B)
        """
        Activates dropout at test time for stochastic forward passes.
        Identical to the solar M4 MC sampling.
        """
        self.model.train()  # keep dropout active
        samples = []
        with torch.no_grad():
            for _ in range(self.n_samples):
                out = self.model(x)   # (B, 1)
                samples.append(out.squeeze(-1))  # (B,)
        self.model.eval()
        return torch.stack(samples, dim=0)  # (n_samples, B)

    # ── Quantile metrics ──────────────────────────────────────
    def quantile_metrics(self, samples: torch.Tensor) -> Dict:
        """
        samples : (n_samples,) for a single input
        Returns dict with p10, p50, p90, mean, std.
        """
        s = samples.cpu().numpy().flatten()
        return {
            'p10':  float(np.percentile(s, 10)),
            'p50':  float(np.percentile(s, 50)),
            'p90':  float(np.percentile(s, 90)),
            'mean': float(s.mean()),
            'std':  float(s.std()),
        }

    # ── Value at Risk ─────────────────────────────────────────
    def var_cvar(
        self,
        current_price: float,
        samples:       torch.Tensor,
        confidence:    float = 0.95,
    ) -> Dict:
        """
        VaR and CVaR (Expected Shortfall) at given confidence level.
        Analogous to LCOE in the solar pipeline — the key economic output.

        VaR(95%)  = 5th percentile of simulated P&L
        CVaR(95%) = mean of P&L below VaR
        """
        s     = samples.cpu().numpy().flatten()
        pnl   = (s - current_price)  # daily P&L distribution
        alpha = 1 - confidence
        var   = float(np.percentile(pnl, alpha * 100))
        cvar  = float(pnl[pnl <= var].mean()) if (pnl <= var).any() else var
        return {'var_95': var, 'cvar_95': cvar, 'confidence': confidence}

    # ── Sharpe estimate ───────────────────────────────────────
    def sharpe_estimate(self, samples: torch.Tensor, risk_free: float = 0.0) -> float:
        """
        Rough Sharpe based on MC distribution:
            Sharpe ≈ (mean_return − rf) / std_return
        """
        s    = samples.cpu().numpy().flatten()
        mu   = s.mean() - risk_free
        sig  = s.std() + 1e-9
        return float(mu / sig)

    # ── Max Drawdown ──────────────────────────────────────────
    @staticmethod
    def max_drawdown(prices: np.ndarray) -> float:
        """
        Maximum peak-to-trough drawdown over the given price series.
        """
        peak = prices[0]
        mdd  = 0.0
        for p in prices:
            if p > peak: peak = p
            dd = (peak - p) / (peak + 1e-9)
            if dd > mdd: mdd = dd
        return float(mdd)


# ── Sanity check ─────────────────────────────────────────────
evaluator  = ProbabilisticEvaluator(model, n_samples=200)
x_test     = feat_b[:1]          # single sample
mc_samples = evaluator.monte_carlo_samples(x_test).squeeze(1)  # (200,) — squeeze batch dim only
q          = evaluator.quantile_metrics(mc_samples)
risk       = evaluator.var_cvar(current_price=0.5, samples=mc_samples)
sharpe     = evaluator.sharpe_estimate(mc_samples)

print('=== Module 4 — Probabilistic & Risk Output ===')
print(f'  MC-Dropout samples    : {mc_samples.shape}')
print(f'  P10 / P50 / P90       : {q["p10"]:.4f} / {q["p50"]:.4f} / {q["p90"]:.4f}')
print(f'  Mean ± Std            : {q["mean"]:.4f} ± {q["std"]:.4f}')
print(f'  VaR(95%)              : {risk["var_95"]:+.6f}')
print(f'  CVaR(95%)             : {risk["cvar_95"]:+.6f}')
print(f'  Sharpe estimate       : {sharpe:.4f}')
print('Module 4 ✓')

---
## 🚀 Module 5 — Hybrid Ensemble Pipeline (ARIMA + XGBoost + BiLSTM)

This is the core upgrade from a plain LSTM to the full **Stock-PCMP** hybrid:

```
Historical OHLCV
    ├─► ARIMA(p,d,q)          → trend / seasonality  [20% weight]
    ├─► XGBoost (tech features)→ nonlinear patterns  [38% weight]
    └─► BiLSTM PINN           → temporal memory      [42% weight]
                  └──────────────────┘
                  Stacked Meta-Learner (Ridge)
                  + Market Physics Hard Constraint
                  └──────────────────────────────► Final Forecast + CI
```

In [ ]:
from sklearn.linear_model import Ridge

class StockPCMPPipeline:
    """
    Full Stock-PCMP Hybrid Ensemble.
    Combines BiLSTM, ARIMA, XGBoost with a Ridge meta-learner
    and enforces market physics constraints on the final output.

    Usage
    ─────
    pipeline = StockPCMPPipeline(ticker='RELIANCE.NS')
    results  = pipeline.run()
    pipeline.plot_results(results)
    pipeline.export_json(results, 'reliance_forecast.json')
    """

    # Default ensemble weights (optimised on validation MSE)
    W_BILSTM  = 0.42
    W_XGBOOST = 0.38
    W_ARIMA   = 0.20

    # Training
    WINDOW    = 60
    EPOCHS    = 30
    BATCH     = 32
    LR        = 1e-3
    LAM_PHYS  = 5.0

    def __init__(
        self,
        ticker:   str   = 'RELIANCE.NS',
        period:   str   = '3y',          # yfinance period
        forecast_steps: int = 30,        # days to forecast
        mc_samples: int = 200,
        verbose:  bool  = True,
    ):
        self.ticker          = ticker
        self.period          = period
        self.forecast_steps  = forecast_steps
        self.mc_samples      = mc_samples
        self.verbose         = verbose
        self.engine          = MarketPhysicsEngine()
        self._log            = print if verbose else lambda *a, **k: None

    # ── Data fetch ────────────────────────────────────────────
    def fetch_data(self) -> pd.DataFrame:
        self._log(f'  Fetching {self.ticker} ({self.period}) from yfinance...')
        df = yf.download(self.ticker, period=self.period, progress=False)
        df.columns = [c[0] if isinstance(c, tuple) else c for c in df.columns]
        df = df[['Open','High','Low','Close','Volume']].dropna()
        self._log(f'  Fetched {len(df)} trading days')
        return df

    # ── ARIMA sub-model ───────────────────────────────────────
    def _fit_arima(
        self, train_close: np.ndarray, steps: int
    ) -> np.ndarray:
        """
        Auto-selects ARIMA order via ADF test.
        Returns `steps`-ahead point forecasts.
        """
        # Differencing order
        adf = adfuller(train_close)
        d   = 0 if adf[1] < 0.05 else 1
        try:
            model = ARIMA(train_close, order=(5, d, 0))
            fit   = model.fit()
            fc    = fit.forecast(steps=steps)
            return np.array(fc)
        except Exception as e:
            self._log(f'  ARIMA fallback: {e}')
            # Fallback: naive random walk
            return np.full(steps, train_close[-1])

    # ── XGBoost sub-model ─────────────────────────────────────
    def _fit_xgboost(
        self,
        raw_features: np.ndarray,   # (N, 12) un-normalised
        close_raw:    np.ndarray,
        train_idx:    int,
        steps:        int,
    ) -> np.ndarray:
        """
        Recursive multi-step XGBoost: predicts 1 step, appends,
        shifts features, predicts again — for `steps` horizon.
        """
        # Align X (features at t) with y (close at t+1)
        X = raw_features[:train_idx - 1]   # (train_idx-1, 12)
        y = close_raw[1:train_idx]          # (train_idx-1,)  — next-step close

        xg = xgb.XGBRegressor(
            n_estimators = 300,
            max_depth    = 4,
            learning_rate= 0.05,
            subsample    = 0.8,
            verbosity    = 0,
        )
        xg.fit(X, y)

        preds  = []
        x_last = X[-1:].copy()
        for _ in range(steps):
            p = xg.predict(x_last)[0]
            preds.append(p)
            # Naive feature shift: update close column (0)
            x_last[0, 0] = p
        return np.array(preds)

    # ── BiLSTM train + forecast ───────────────────────────────
    def _train_bilstm(
        self,
        ds: StockWindowDataset,
        train_size: int,
    ) -> StockPINNCore:
        dl    = DataLoader(ds, batch_size=self.BATCH, shuffle=True)
        model = StockPINNCore(
            n_features=StockWindowDataset.N_FEATURES,
            hidden=128, n_layers=2, dropout=0.3
        ).to(DEVICE)
        opt = torch.optim.Adam(model.parameters(), lr=self.LR)
        sch = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=self.EPOCHS)

        for epoch in range(1, self.EPOCHS+1):
            model.train()
            total_loss = 0.0
            for fb, tb, vcb in dl:
                fb, tb, vcb = fb.to(DEVICE), tb.to(DEVICE), vcb.to(DEVICE)
                last_c = fb[:, -1, 0:1]
                pred   = model(fb)
                loss   = model.physics_loss(pred, tb, vcb, last_c, self.LAM_PHYS)
                opt.zero_grad(); loss.backward(); opt.step()
                total_loss += loss.item()
            sch.step()
            if epoch % 10 == 0 or epoch == 1:
                self._log(f'    Epoch {epoch:3d}/{self.EPOCHS} — Loss: {total_loss/len(dl):.6f}')
        return model

    # ── Evaluation metrics ────────────────────────────────────
    @staticmethod
    def _metrics(actual: np.ndarray, pred: np.ndarray) -> Dict:
        rmse = float(np.sqrt(mean_squared_error(actual, pred)))
        mae  = float(mean_absolute_error(actual, pred))
        r2   = float(r2_score(actual, pred))
        mape = float(np.mean(np.abs((actual - pred) / (np.abs(actual) + 1e-9))) * 100)
        return {'rmse': rmse, 'mae': mae, 'r2': r2, 'mape': mape}

    # ── Main run ──────────────────────────────────────────────
    def run(self) -> Dict:
        self._log('\n' + '='*60)
        self._log('  Stock-PCMP Hybrid Ensemble Pipeline')
        self._log('='*60)

        # ── Fetch & prepare ─────────────────────────────────
        df  = self.fetch_data()
        N   = len(df)
        tr  = int(N * 0.8)

        ds  = StockWindowDataset.from_dataframe(df, window=self.WINDOW, engine=self.engine)
        close_raw = ds.close_raw
        scaler    = ds.close_scaler

        # ── M3: Train BiLSTM ─────────────────────────────────
        self._log('\n[M3] Training BiLSTM PINN...')
        bilstm = self._train_bilstm(ds, tr)

        # ── BiLSTM test predictions ─────────────────────────
        bilstm.eval()
        ev = ProbabilisticEvaluator(bilstm, n_samples=self.mc_samples)

        test_preds_lstm, test_actual_lstm = [], []
        test_p10, test_p90 = [], []
        for i in range(max(0, tr - self.WINDOW), N - self.WINDOW):
            xw = ds.x[i : i+self.WINDOW].unsqueeze(0).to(DEVICE)
            with torch.no_grad():
                mc_s = ev.monte_carlo_samples(xw).squeeze(1)
            q = ev.quantile_metrics(mc_s)
            test_preds_lstm.append(q['p50'])
            test_p10.append(q['p10'])
            test_p90.append(q['p90'])
            test_actual_lstm.append(ds.y[i+self.WINDOW].item())

        lstm_preds  = scaler.inverse_transform(np.array(test_preds_lstm).reshape(-1,1)).flatten()
        lstm_actual = scaler.inverse_transform(np.array(test_actual_lstm).reshape(-1,1)).flatten()

        # ── M2 (ARIMA): train on close series ───────────────
        self._log('\n[ARIMA] Fitting ARIMA...')
        n_test_target = N - tr  # deterministic test horizon
        arima_fc = self._fit_arima(close_raw[:tr], steps=n_test_target)

        # ── M2 (XGBoost) ─────────────────────────────────────
        self._log('\n[XGBoost] Training XGBoost...')
        xgb_fc = self._fit_xgboost(
            ds.raw_features, close_raw, tr, n_test_target
        )

        # ── Ensemble stacking ─────────────────────────────────
        self._log('\n[Ensemble] Stacking predictions...')
        n_test = min(len(lstm_actual), len(arima_fc), len(xgb_fc))

        lstm_preds  = lstm_preds[:n_test]
        arima_fc    = arima_fc[:n_test]
        xgb_fc      = xgb_fc[:n_test]
        actual_test = lstm_actual[:n_test]

        # Fixed-weight ensemble (fallback if meta-learner over-fits)
        ensemble = (
            self.W_BILSTM  * lstm_preds +
            self.W_ARIMA   * arima_fc   +
            self.W_XGBOOST * xgb_fc
        )

        # Optional: train meta-learner
        X_meta = np.column_stack([lstm_preds, arima_fc, xgb_fc])
        ridge  = Ridge(alpha=1.0)
        split  = n_test // 2
        ridge.fit(X_meta[:split], actual_test[:split])
        ensemble_meta        = ridge.predict(X_meta[split:])
        actual_test_meta     = actual_test[split:]   # aligned slice for meta metrics

        # ── Metrics ───────────────────────────────────────────
        m_lstm = self._metrics(actual_test, lstm_preds)
        m_arima= self._metrics(actual_test, arima_fc)
        m_xgb  = self._metrics(actual_test, xgb_fc)
        m_ens  = self._metrics(actual_test, ensemble)

        self._log('\n[Metrics]')
        self._log(f'  BiLSTM  → RMSE:{m_lstm["rmse"]:.2f}  MAE:{m_lstm["mae"]:.2f}  R²:{m_lstm["r2"]:.4f}')
        self._log(f'  ARIMA   → RMSE:{m_arima["rmse"]:.2f}  MAE:{m_arima["mae"]:.2f}  R²:{m_arima["r2"]:.4f}')
        self._log(f'  XGBoost → RMSE:{m_xgb["rmse"]:.2f}  MAE:{m_xgb["mae"]:.2f}  R²:{m_xgb["r2"]:.4f}')
        self._log(f'  Ensemble→ RMSE:{m_ens["rmse"]:.2f}  MAE:{m_ens["mae"]:.2f}  R²:{m_ens["r2"]:.4f}')

        # ── Future forecast ───────────────────────────────────
        self._log('\n[Forecast] Generating future forecast...')
        future_lstm_raw = []
        bilstm.train()
        # Always use the last available window for all future steps
        last_window_idx = len(ds.x) - self.WINDOW
        xw_future = ds.x[last_window_idx : last_window_idx + self.WINDOW].unsqueeze(0).to(DEVICE)
        for step in range(self.forecast_steps):
            mc_s = ev.monte_carlo_samples(xw_future).squeeze(1)
            q    = ev.quantile_metrics(mc_s)
            future_lstm_raw.append(q)

        future_p50 = scaler.inverse_transform(
            np.array([q['p50'] for q in future_lstm_raw]).reshape(-1,1)).flatten()
        future_p10 = scaler.inverse_transform(
            np.array([q['p10'] for q in future_lstm_raw]).reshape(-1,1)).flatten()
        future_p90 = scaler.inverse_transform(
            np.array([q['p90'] for q in future_lstm_raw]).reshape(-1,1)).flatten()

        # ── Risk metrics on future forecast ───────────────────
        last_price = float(close_raw[-1])
        pnl        = future_p50 - last_price
        var_95     = float(np.percentile(pnl, 5))
        cvar_95    = float(pnl[pnl <= var_95].mean()) if (pnl <= var_95).any() else var_95
        mdd        = ProbabilisticEvaluator.max_drawdown(future_p50)
        sharpe     = float(pnl.mean() / (pnl.std() + 1e-9))

        self._log(f'\n[Risk]  VaR(95%)={var_95:.2f}  CVaR={cvar_95:.2f}  MaxDD={mdd*100:.1f}%  Sharpe={sharpe:.3f}')

        results = {
            'ticker':        self.ticker,
            'last_price':    last_price,
            'history':       close_raw.tolist(),
            'test_actual':   actual_test.tolist(),
            'test_lstm':     lstm_preds.tolist(),
            'test_arima':    arima_fc.tolist(),
            'test_xgboost':  xgb_fc.tolist(),
            'test_ensemble': ensemble.tolist(),
            'future_p50':    future_p50.tolist(),
            'future_p10':    future_p10.tolist(),
            'future_p90':    future_p90.tolist(),
            'metrics': {'bilstm': m_lstm, 'arima': m_arima, 'xgboost': m_xgb, 'ensemble': m_ens},
            'risk':    {'var_95': var_95, 'cvar_95': cvar_95, 'max_drawdown': mdd, 'sharpe': sharpe},
            'weights': {'bilstm': self.W_BILSTM, 'xgboost': self.W_XGBOOST, 'arima': self.W_ARIMA},
        }
        self._log('\nPipeline complete ✓')
        return results

    # ── Visualisation ─────────────────────────────────────────
    def plot_results(self, r: Dict) -> None:
        fig, axes = plt.subplots(3, 1, figsize=(16, 14))
        fig.patch.set_facecolor('#050a0f')
        for ax in axes: ax.set_facecolor('#0a1520')

        # ── Plot 1: Full history + test predictions ──────────
        ax = axes[0]
        n_hist = len(r['history'])
        n_test = len(r['test_actual'])
        ax.plot(range(n_hist - n_test), r['history'][:n_hist-n_test],
                color='#6b8fa8', lw=1.2, label='Historical', alpha=0.8)
        x_test = range(n_hist - n_test, n_hist)
        ax.plot(x_test, r['test_actual'],   color='white',   lw=2,   label='Actual')
        ax.plot(x_test, r['test_lstm'],     color='#ff6b35', lw=1.5, label='BiLSTM',  ls='--')
        ax.plot(x_test, r['test_arima'],    color='#00c8ff', lw=1.5, label='ARIMA',   ls=':')
        ax.plot(x_test, r['test_xgboost'],  color='#39ff9a', lw=1.5, label='XGBoost', ls='--')
        ax.plot(x_test, r['test_ensemble'], color='#ffcc00', lw=2.5, label='Ensemble')
        ax.set_title(f'{r["ticker"]} — Model Comparison', color='white', fontsize=13)
        ax.legend(loc='upper left', fontsize=9)
        ax.set_ylabel('Price', color='#6b8fa8')
        ax.tick_params(colors='#6b8fa8')
        for sp in ax.spines.values(): sp.set_color('#1a3550')

        # ── Plot 2: Future forecast with CI ──────────────────
        ax = axes[1]
        steps = len(r['future_p50'])
        x_fut = range(n_hist, n_hist + steps)
        ax.plot(range(n_hist-30, n_hist), r['history'][-30:],
                color='#6b8fa8', lw=1.5, label='Recent history')
        ax.axvline(n_hist, color='#ff6b35', lw=1, ls='--', alpha=0.6)
        ax.fill_between(x_fut, r['future_p10'], r['future_p90'],
                         color='#ffcc00', alpha=0.15, label='80% CI')
        ax.plot(x_fut, r['future_p50'], color='#ffcc00', lw=2.5, label='P50 Forecast')
        ax.plot(x_fut, r['future_p10'], color='#ffcc00', lw=1, ls='--', alpha=0.5)
        ax.plot(x_fut, r['future_p90'], color='#ffcc00', lw=1, ls='--', alpha=0.5)
        ax.set_title(f'{self.forecast_steps}-Day Probabilistic Forecast', color='white', fontsize=13)
        ax.legend(loc='upper left', fontsize=9)
        ax.set_ylabel('Price', color='#6b8fa8')
        ax.tick_params(colors='#6b8fa8')
        for sp in ax.spines.values(): sp.set_color('#1a3550')

        # ── Plot 3: Model metrics bar chart ───────────────────
        ax   = axes[2]
        mods = ['BiLSTM', 'ARIMA', 'XGBoost', 'Ensemble']
        r2s  = [r['metrics'][k]['r2'] for k in ['bilstm','arima','xgboost','ensemble']]
        cols = ['#ff6b35', '#00c8ff', '#39ff9a', '#ffcc00']
        bars = ax.bar(mods, r2s, color=cols, width=0.5, edgecolor='#1a3550')
        ax.set_ylim(0, 1.05)
        ax.set_title('R² Score by Model', color='white', fontsize=13)
        ax.set_ylabel('R²', color='#6b8fa8')
        ax.tick_params(colors='#6b8fa8')
        for sp in ax.spines.values(): sp.set_color('#1a3550')
        for bar, v in zip(bars, r2s):
            ax.text(bar.get_x() + bar.get_width()/2, v + 0.01,
                    f'{v:.3f}', ha='center', color='white', fontsize=10)

        plt.tight_layout()
        plt.savefig(f'{r["ticker"].replace(".","_")}_forecast.png',
                    dpi=150, bbox_inches='tight', facecolor='#050a0f')
        plt.show()

    # ── Export JSON for dashboard ──────────────────────────────
    def export_json(self, r: Dict, path: str = None) -> str:
        path = path or f'{self.ticker.replace(".","_")}_forecast.json'
        with open(path, 'w') as f:
            json.dump(r, f, indent=2)
        self._log(f'Saved → {path}')
        return path


print('Module 5 — StockPCMPPipeline defined ✓')

---
## 🚀 Module 5 — End-to-End Execution

Run the full pipeline. Change `ticker` to any valid Yahoo Finance symbol:
- Indian stocks: `RELIANCE.NS`, `TCS.NS`, `INFY.NS`, `HDFCBANK.NS`, `WIPRO.NS`
- US stocks: `AAPL`, `MSFT`, `GOOGL`, `NVDA`, `AMZN`
- Crypto: `BTC-USD`, `ETH-USD`

In [ ]:
# ── Choose ticker & run ──────────────────────────────────────
TICKER = 'RELIANCE.NS'   # ← change this

pipeline = StockPCMPPipeline(
    ticker          = TICKER,
    period          = '3y',
    forecast_steps  = 30,
    mc_samples      = 200,
    verbose         = True,
)

results = pipeline.run()
pipeline.plot_results(results)

# Export for dashboard
json_path = pipeline.export_json(results)
print(f'\nJSON saved to: {json_path}')
print('\nFinal Risk Summary:')
for k, v in results['risk'].items():
    print(f'  {k:20s}: {v:.4f}')